# 📘 Tutorial 4: Choosing Where to Evaluate Next

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

In **Tutorial 3**, we studied how a Gaussian Process turns sparse observations into a principled surrogate model with:

- a predictive mean,
- a predictive uncertainty,
- and a full probabilistic interpretation through the GP posterior.

We saw that once the objective is only partially observed:
- prediction and uncertainty must be modelled together,
- observations update the whole distribution over possible functions,
- and the surrogate becomes a dynamic object that changes as data are added.

In this tutorial, we take the next conceptual step:

> **given a Gaussian Process surrogate, where should we evaluate next?**

This is the central question of **Bayesian Optimisation**.

It marks another important shift.

In the previous tutorial, the Gaussian Process itself was the main object of study.
Here, the GP is no longer the endpoint.
It becomes the **input to a decision rule**.

Once we have a surrogate with

$$
\mu(x)
\qquad\text{and}\qquad
\sigma(x),
$$

we still need a way to decide which candidate point is most worth evaluating next.

That means Bayesian Optimisation is not just about modelling the unknown objective.

It is also about:
- turning prediction and uncertainty into a decision,
- balancing exploitation against exploration,
- defining acquisition functions that score candidate points,
- and updating the surrogate sequentially as new evaluations are collected.

To make this concrete, we continue with simple one-dimensional black-box objectives and study how different acquisition rules behave when built from the same Gaussian Process surrogate.

---

**This tutorial is designed to shift perspective**
- from *“a Gaussian Process tells us what the model believes”*
- to *“Bayesian Optimisation uses that belief to decide what information is most valuable next.”*

---

**The emphasis is on developing intuition for**
- why minimising the posterior mean alone is too greedy,
- how acquisition functions use both prediction and uncertainty,
- why PI and EI are different notions of “promise,”
- how the next selected point changes the surrogate,
- and how Bayesian Optimisation emerges as a sequential loop of modelling, scoring, evaluating, and updating.

---

**Key ideas explored include**
- mean-only selection as pure exploitation,
- acquisition functions as next-point decision rules,
- Lower Confidence Bound (LCB),
- Probability of Improvement (PI),
- Expected Improvement (EI),
- the effect of adding PI- and EI-selected points,
- and the full Bayesian Optimisation loop over multiple iterations.

---

This tutorial serves as the conceptual bridge from **Gaussian Process surrogate modelling** to **Bayesian Optimisation as a sequential optimisation strategy**.

In particular, it shows that once the surrogate provides both a predictive mean and a predictive uncertainty:
- optimisation becomes a problem of choosing where information is most valuable,
- acquisition functions become the decision layer built on top of the GP,
- and the model and the data begin to co-evolve through repeated evaluation and updating.

These ideas prepare the ground for the next part of the repository, where we move from concept-first BO implementations to more practical workflows using **BoTorch**.

---

**Recommended prerequisites**
- Completion of **Tutorials 1–3** in Part 3
- Basic familiarity with Gaussian Processes and predictive uncertainty
- Comfort with one-dimensional optimisation plots and sequential model updates

---

**Author**: Angze Li

**Last updated**: 2026-04-02

**Version**: v1.0

## 🔧 Setup

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt, pi

torch.set_default_dtype(torch.float64)

def set_seed(seed=0):
    torch.manual_seed(seed)
    np.random.seed(seed)

def style_ax(ax):
    for spine in ax.spines.values():
        spine.set_linewidth(1.8)
    ax.tick_params(axis="both", labelsize=14)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontweight("bold")

set_seed(0)

In [ ]:
def expensive_objective(x):
    return 0.35 * torch.sin(2.2 * x) + 0.15 * torch.cos(5.0 * x) + 0.03 * (x - 1.5) ** 2 - 0.25

def rbf_kernel(x1, x2, lengthscale=0.8, variance=1.0):
    x1 = x1.reshape(-1, 1)
    x2 = x2.reshape(-1, 1)
    sqdist = (x1 - x2.T) ** 2
    return variance * torch.exp(-0.5 * sqdist / lengthscale**2)

def gp_posterior(x_train, y_train, x_test, lengthscale=0.8, variance=1.0, noise=1e-4):
    K_xx = rbf_kernel(x_train, x_train, lengthscale=lengthscale, variance=variance)
    K_xs = rbf_kernel(x_train, x_test, lengthscale=lengthscale, variance=variance)
    K_ss = rbf_kernel(x_test, x_test, lengthscale=lengthscale, variance=variance)

    K_xx = K_xx + noise * torch.eye(len(x_train), dtype=x_train.dtype)

    L = torch.linalg.cholesky(K_xx)
    alpha = torch.cholesky_solve(y_train.unsqueeze(1), L)

    mu = (K_xs.T @ alpha).squeeze()

    v = torch.linalg.solve_triangular(L, K_xs, upper=False)
    cov = K_ss - v.T @ v
    var = torch.clamp(torch.diag(cov), min=1e-12)

    return mu, var, cov

x_dense = torch.linspace(-3.0, 3.0, 400)
y_dense = expensive_objective(x_dense)

## 1. Conceptual setup: the current Gaussian Process surrogate

We begin this tutorial from the same Gaussian Process surrogate developed in **Tutorial 3**.

There, the main goal was to understand how a GP turns sparse observations into two key objects:

$$
\mu(x)
\qquad\text{and}\qquad
\sigma(x),
$$

that is:

- a predictive mean
- and a predictive uncertainty

In this tutorial, those quantities are no longer the endpoint.
They are now the starting point for a new question:

> **Given this surrogate, where should we evaluate next?**

So this cell serves as a conceptual reminder of the current modelling state before we introduce acquisition functions and Bayesian Optimisation decisions.

---

### What the code does

The code begins by constructing a small training dataset of 8 observations in the interval

$$
[-1.5,\;1.5].
$$

It then computes the Gaussian Process posterior over the dense grid `x_dense`, producing:

- the posterior mean
  $$
  \mu(x),
  $$
- and the posterior standard deviation
  $$
  \sigma(x).
  $$

The figure shows:

- the true objective, for reference
- the GP mean prediction
- the uncertainty band
  $$
  \mu(x)\pm 2\sigma(x)
  $$
- and the observed data points

So this plot is the current surrogate model that Bayesian Optimisation will act on.

---

### How to interpret the figure

This figure should now feel familiar from the previous tutorial.

The dashed GP mean represents the model’s current best estimate of the unknown objective, while the shaded uncertainty band shows where the model is more or less confident.

Near the observed points, the model is better constrained.
Farther away from those points, uncertainty remains larger.

So the figure gives a complete snapshot of the model’s current state of knowledge:

- what the model believes
- and how certain it is about that belief

That is exactly the information we need in order to decide where to evaluate next.

---

### Why this matters for Tutorial 4

This is the key transition into Bayesian Optimisation.

In Tutorial 3, the GP was the main object of study.
In Tutorial 4, the GP becomes the **input** to a decision rule.

The challenge is no longer just to model the unknown function.

It is to use the surrogate intelligently.

That means we now need a way to turn

$$
\mu(x)
\qquad\text{and}\qquad
\sigma(x)
$$

into an actual choice of the next evaluation point.

That is the role of an **acquisition function**, which is the central idea of the rest of this notebook.

---

### Key takeaway

This figure is the starting point for Bayesian Optimisation:

> a Gaussian Process surrogate gives us both a prediction and an uncertainty estimate, and the next step is to use those two quantities to decide where to sample next.

In [ ]:
x_train = torch.linspace(-1.5, 1.5, 8)
y_train = expensive_objective(x_train)

mu_gp, var_gp, _ = gp_posterior(x_train, y_train, x_dense, lengthscale=0.8, variance=1.0, noise=1e-4)
sigma_gp = torch.sqrt(var_gp)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
ax.plot(x_dense.numpy(), mu_gp.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
ax.fill_between(
    x_dense.numpy(),
    (mu_gp - 2.0 * sigma_gp).numpy(),
    (mu_gp + 2.0 * sigma_gp).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)
ax.scatter(x_train.numpy(), y_train.numpy(), s=40, zorder=3, label="Observations")
ax.set_title("Current Gaussian Process surrogate", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 2. Choosing by posterior mean alone is a purely exploitative rule

Now that we have a Gaussian Process surrogate, the next question is:

> **Where should we evaluate next?**

A natural first idea is to choose the point where the GP posterior mean is smallest.

That means selecting

$$
x_{\text{next}} = \arg\min_x \mu(x),
$$

where $$\mu(x)$$ is the current GP mean prediction.

This is the most direct way to turn the surrogate into a decision rule.

If the GP mean represents the model’s current best estimate of the unknown objective, then it is tempting to say:

> “Just evaluate where the model currently thinks the objective is lowest.”

This cell shows why that idea is useful, but also why it is incomplete.

---

### What the code does

The code finds the input location where the GP posterior mean attains its minimum:

$$
x_{\text{next}} = \arg\min_x \mu(x).
$$

It then plots:

- the true objective, for reference,
- the GP mean prediction,
- the uncertainty band $\mu(x)\pm 2\sigma(x)$,
- the observed data points,
- and the selected next point under this **mean-only rule**.

So the figure answers the question:

> **What point would we choose next if we only trusted the GP mean?**

---

### How to interpret the figure

The selected point is where the GP mean is lowest.

This makes the decision rule very easy to understand:

- low predicted value looks good,
- so choose the point with the smallest current prediction.

This is a perfectly reasonable baseline, and in some cases it may work well.

But it has a major limitation:

> it only uses $\mu(x)$ and ignores $\sigma(x)$.

That means it is a **pure exploitation strategy**.

It assumes that the current mean prediction is already the only thing that matters.

---

### Why this is not enough

The problem is that the GP mean is only the model’s current best estimate.

It does not tell the whole story.

A region may have:
- a moderately low mean,
- but very high uncertainty,

which means the true objective there could still turn out to be much better than the current mean suggests.

A mean-only rule ignores that possibility.

So this decision rule can become too greedy:

- it focuses on what currently looks best,
- but it may fail to explore uncertain regions that could contain better solutions.

That is exactly the tension at the heart of Bayesian Optimisation.

---

### The exploration–exploitation issue

This figure is the first point in the notebook where the exploration–exploitation trade-off becomes explicit.

Choosing by posterior mean alone corresponds to:

- **high exploitation**
- **no explicit exploration**

It asks:

> where does the model already think the function is low?

but it does not ask:

> where is the model still uncertain in a potentially useful way?

That missing second question is the reason Bayesian Optimisation needs **acquisition functions** rather than just the posterior mean.

---

### Why the true objective is shown here

In real optimisation, we would not know the full true objective everywhere.

It is plotted here only for teaching purposes, so that we can compare:

- what the GP currently believes,
- and how that belief relates to the actual function.

This makes it easier to see why a mean-only choice can be too myopic.

So the true objective is not part of the decision rule.
It is included only to help interpret the behaviour of that rule.

---

### Key takeaway

> Choosing the next evaluation point by minimising the GP mean alone is a purely exploitative strategy.

It uses the surrogate’s current prediction, but it ignores uncertainty.

That makes it a natural baseline, but not yet a full Bayesian Optimisation rule.

The next step is therefore to introduce acquisition functions that combine:

- predicted value
- and predictive uncertainty

so that the next point can be chosen by balancing **exploitation** with **exploration**.

In [ ]:
idx_mean = torch.argmin(mu_gp)
x_next_mean = x_dense[idx_mean]
y_next_mean = mu_gp[idx_mean]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
ax.plot(x_dense.numpy(), mu_gp.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
ax.fill_between(
    x_dense.numpy(),
    (mu_gp - 2.0 * sigma_gp).numpy(),
    (mu_gp + 2.0 * sigma_gp).numpy(),
    alpha=0.2,
    label=r"$\mu(x)\pm 2\sigma(x)$"
)
ax.scatter(x_train.numpy(), y_train.numpy(), s=40, zorder=3, label="Observations")
ax.scatter([float(x_next_mean)], [float(y_next_mean)], s=90, marker="*", zorder=4, label="Mean-only choice")
ax.set_title("Choosing by posterior mean alone", fontsize=18, fontweight="bold")
ax.set_xlabel("x", fontsize=22, fontweight="bold")
ax.set_ylabel("Prediction", fontsize=22, fontweight="bold")
ax.legend(prop={"size": 10, "weight": "bold"})
style_ax(ax)
plt.show()

## 3. Defining acquisition functions from the GP posterior

The previous cell showed that choosing the next point by minimising the GP mean alone is a purely exploitative strategy.

That naturally raises the next question:

> **How can we define decision rules that use both prediction and uncertainty?**

This is the role of an **acquisition function**.

An acquisition function is a scoring rule built from the current Gaussian Process surrogate.
It does not model the objective directly.
Instead, it tells us how attractive each candidate input is as the **next evaluation point**.

So if the GP gives the model,

$$
\mu(x)
\qquad\text{and}\qquad
\sigma(x),
$$

then the acquisition function tells us what to do with them.

In practice, once an acquisition function has been computed across the candidate domain, the **next observation point** is chosen by optimising that acquisition function.

So for the rules introduced here:

- for **LCB**, we choose the point that **minimises** `LCB(x)`;
- for **PI**, we choose the point that **maximises** `PI(x)`;
- for **EI**, we choose the point that **maximises** `EI(x)`.

This means that the maxima of PI and EI are interpreted as the most attractive next points to evaluate under those decision rules.

---


### What the code does

This cell defines the quantities needed for three standard acquisition rules:

- **Lower Confidence Bound (LCB)**
- **Probability of Improvement (PI)**
- **Expected Improvement (EI)**

It begins by computing

$$
y_{\text{best}} = \min(y_{\text{train}}),
$$

which is the best objective value *observed* (not predicted) so far.

This quantity is important because improvement-based acquisition functions need a benchmark:
a candidate point is attractive if it is likely to beat the current best observed value.

The code then defines:

- the standard normal probability density function
- the standard normal cumulative distribution function
- and the three acquisition functions themselves

These functions will be used in the next few cells to compare different ways of choosing the next evaluation.

---

### Why do we need `y_best`?

For minimisation, the quantity

$$
y_{\text{best}} = \min(y_{\text{train}})
$$

represents the best value we have found so far.

Improvement-based acquisition functions ask questions like:

- how likely is a candidate point to improve on this?
- by how much might it improve on this?

So `y_best` is the natural reference level against which future candidate points are judged.

This is why PI and EI are built relative to the current best observed value rather than only the GP mean.

---

### Lower Confidence Bound (LCB)

The first acquisition rule is

$$
\text{LCB}(x) = \mu(x) - \beta \sigma(x),
$$

where $\beta \ge 0$ controls how strongly uncertainty is rewarded.

This is the same exploration–exploitation idea that appeared earlier in simplified form:

- low $\mu(x)$ favours **exploitation**
- high $\sigma(x)$ favours **exploration**

So LCB says:

> choose points that either look good already or are uncertain enough to be worth investigating.

A small $\beta$ gives a more exploitative rule.
A larger $\beta$ makes the method more exploratory.

---

### Probability of Improvement (PI)

The second acquisition rule is **Probability of Improvement**.

For minimisation, PI measures the probability that a candidate point will produce a value below the current best:

$$
\text{PI}(x) = \Phi\!\left(\frac{y_{\text{best}} - \mu(x) - \xi}{\sigma(x)}\right),
$$

where:

- $\Phi$ is the standard normal cumulative distribution function (CDF)
- $\xi$ is a small optional improvement threshold

So PI asks:

> **How likely is this point to beat the best value we have already seen?**

This makes PI more explicitly improvement-based than LCB.

However, PI only measures the probability of improvement, not the size of the improvement.

---

### Expected Improvement (EI)

The third acquisition rule is **Expected Improvement**.

For minimisation, EI is

$$
\text{EI}(x)=\bigl(y_{\text{best}} - \mu(x) - \xi\bigr)\Phi(z)+\sigma(x)\phi(z),
$$

where

$$
z = \frac{y_{\text{best}} - \mu(x) - \xi}{\sigma(x)},
$$

and:

- $\Phi$ is the standard normal CDF
- $\phi$ is the standard normal probability distribution function (PDF)

So EI asks a more refined question:

> **How large is the expected improvement over the current best value?**

Unlike PI, EI balances:
- the chance of improvement
- and the magnitude of that improvement

For example, a point with moderate probability of a large improvement can be preferred over a point with high probability of a tiny improvement.

That is why EI is one of the most widely used acquisition functions in Bayesian Optimisation.

---

### Why the normal PDF and CDF appear

The GP posterior at each point is Gaussian:

$$
f(x)\mid \mathcal{D} \sim \mathcal{N}(\mu(x), \sigma^2(x)).
$$

Because PI and EI are based on probabilities and expectations under that Gaussian predictive distribution, they naturally involve:

- the standard normal PDF
  $$
  \phi(z)
  $$
- and the standard normal CDF
  $$
  \Phi(z)
  $$

So the functions `std_normal_pdf` and `std_normal_cdf` are mathematical building blocks used to convert the GP posterior into decision rules.

---

### What this cell means conceptually

This cell is the point where the notebook moves from:

- **modelling** the objective with a Gaussian Process

to

- **making decisions** using that model

That is the core step into Bayesian Optimisation.

The GP itself tells us:
- what we currently believe
- and how uncertain we still are

The acquisition function then turns that information into a ranking of candidate evaluation points.

So this cell introduces the three key decision rules that will drive the rest of the notebook.

---

### Key takeaway

> A Gaussian Process provides prediction and uncertainty, but an acquisition function turns those quantities into a next-point selection rule.

In this cell, we define three important acquisition functions:

- **LCB** for balancing low mean and high uncertainty
- **PI** for measuring the probability of beating the current best value
- **EI** for measuring the expected amount of improvement

These are the core tools that make Bayesian Optimisation an active sequential decision process rather than just a surrogate modelling exercise.

In [ ]:
y_best = torch.min(y_train)

def std_normal_pdf(z):
    return torch.exp(-0.5 * z**2) / sqrt(2.0 * pi)

def std_normal_cdf(z):
    return 0.5 * (1.0 + torch.erf(z / sqrt(2.0)))

def lcb_acquisition(mu, sigma, beta=1.0):
    return mu - beta * sigma

def pi_acquisition(mu, sigma, y_best, xi=0.0):
    z = (y_best - mu - xi) / torch.clamp(sigma, min=1e-12)
    return std_normal_cdf(z)

def ei_acquisition(mu, sigma, y_best, xi=0.0):
    sigma_safe = torch.clamp(sigma, min=1e-12)
    z = (y_best - mu - xi) / sigma_safe
    ei = (y_best - mu - xi) * std_normal_cdf(z) + sigma_safe * std_normal_pdf(z)
    ei = torch.where(sigma > 0, ei, torch.zeros_like(ei))
    return ei

## 4. Probability of Improvement and Expected Improvement

The previous cell defined three acquisition functions from the Gaussian Process posterior.

We now begin by looking more closely at the two **improvement-based** rules:

- **Probability of Improvement (PI)**
- **Expected Improvement (EI)**

These are both built relative to the current best observed value,

$$
y_{\text{best}} = \min(y_{\text{train}}),
$$

but they measure different aspects of what it means for a candidate point to be promising.

This cell visualises those two acquisition functions directly.

---

### What the code does

Using the current GP posterior mean

$$
\mu(x)
$$

and posterior standard deviation

$$
\sigma(x),
$$

the code computes:

- the **Probability of Improvement**
  $$
  \text{PI}(x),
  $$
- and the **Expected Improvement**
  $$
  \text{EI}(x).
  $$

These are evaluated across the dense grid `x_dense` and plotted side by side.

So the figure does **not** show the objective or the GP mean directly.
Instead, it shows the **decision scores** that PI and EI assign to each candidate input.

This is the first point in the notebook where the acquisition functions themselves become the main object of study.

---

### What PI is measuring

Probability of Improvement asks:

> **How likely is this point to beat the best value we have observed so far?**

Mathematically, it computes the probability that the GP-predicted function value at $x$ lies below the current best threshold.

So PI focuses entirely on the **chance of improvement**.

A point receives a high PI score if:
- the GP mean is low relative to the current best,
- or the uncertainty is large enough that improvement is plausible.

However, PI does not care about how *large* that improvement might be.
It only cares about whether improvement happens.

So PI can sometimes favour points that have a decent chance of only a very small improvement.

---

### What EI is measuring

Expected Improvement asks a more refined question:

> **How much improvement should we expect on average if we evaluate at this point?**

This means EI takes into account both:

- the probability that improvement occurs,
- and the size of that improvement if it does occur.

So EI is more nuanced than PI.

A point can receive a high EI score if:
- it already looks very promising under the GP mean,
- or it is uncertain enough that a substantial improvement is still possible.

This is why EI often provides a more balanced exploration–exploitation rule than PI.

---

### How to interpret the two panels

The two panels show how the same GP posterior gives rise to two different acquisition landscapes.

The **PI** panel highlights points that are most likely to beat the current best value.

The **EI** panel highlights points that are expected to produce the greatest average improvement.

So even though PI and EI are both based on improvement, they are not identical:

- PI focuses on **probability**
- EI focuses on **expected magnitude**

This difference often leads them to emphasise different parts of the domain.

---

### Why this matters

This figure marks an important shift in perspective.

Up to now, the Gaussian Process has been used to describe:
- what the model predicts,
- and how uncertain it is.

But Bayesian Optimisation needs more than that.

It needs a rule for turning those quantities into a **decision about where to sample next**.

PI and EI do exactly that.

They are not surrogate models themselves.
They are functions built *from* the surrogate in order to guide sequential evaluation.

So this cell is the point where the notebook moves from:
- understanding the GP posterior
to
- understanding how that posterior is used for optimisation.

---

### Key takeaway

> PI and EI are both improvement-based acquisition functions, but they measure different notions of promise.

- **PI** asks how likely a point is to improve on the current best observed value
- **EI** asks how much improvement that point is expected to deliver on average

This makes them two closely related, but distinct, decision rules for choosing the next evaluation point.

In [ ]:
pi_vals = pi_acquisition(mu_gp, sigma_gp, y_best, xi=0.0)
ei_vals = ei_acquisition(mu_gp, sigma_gp, y_best, xi=0.0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharex=True)

axes[0].plot(x_dense.numpy(), pi_vals.numpy(), linewidth=2.0)
axes[0].set_title("Probability of Improvement", fontsize=18, fontweight="bold")
axes[0].set_xlabel("x", fontsize=22, fontweight="bold")
axes[0].set_ylabel("PI", fontsize=22, fontweight="bold")
style_ax(axes[0])

axes[1].plot(x_dense.numpy(), ei_vals.numpy(), linewidth=2.0)
axes[1].set_title("Expected Improvement", fontsize=18, fontweight="bold")
axes[1].set_xlabel("x", fontsize=22, fontweight="bold")
axes[1].set_ylabel("EI", fontsize=22, fontweight="bold")
style_ax(axes[1])

plt.tight_layout()
plt.show()

## 5. PI and EI do not just choose points — they change the surrogate differently

The previous figure showed the acquisition landscapes of **Probability of Improvement (PI)** and **Expected Improvement (EI)**.

That tells us where each rule prefers to sample.

But a more important question is:

> **What happens to the Gaussian Process surrogate after we actually add the point chosen by PI or EI?**

This cell answers that question directly.

Instead of only marking the selected point, it compares the GP **before** and **after** adding the point chosen by each acquisition rule.

That makes the consequences of the decision visible.

---

### What the code does

The code first finds:

- the point that maximises `PI(x)`
- and the point that maximises `EI(x)`

These become the next candidate evaluation points under the two acquisition rules.

For each rule separately, the code then:

1. takes the selected point
2. evaluates the true objective there
3. appends that new observation to the training data
4. recomputes the GP posterior mean and uncertainty

The figure is arranged as a $2\times2$ grid:

- **left column**: PI
- **right column**: EI
- **top row**: before adding the selected point
- **bottom row**: after adding the selected point

So this figure shows not just the next-point choice, but the actual effect of that choice on the surrogate.

---

### How to interpret the top row

The top row shows the original GP surrogate before any new point has been added.

In each panel:

- the dashed curve is the current GP mean
- the shaded region is the uncertainty band
  $$
  \mu(x)\pm 2\sigma(x)
  $$
- the star marks the point selected by the corresponding acquisition rule

So the top row answers:

- where would PI choose next?
- where would EI choose next?

This makes the decision itself visible.

---

### How to interpret the bottom row

The bottom row shows the GP posterior after the selected point has actually been evaluated and incorporated into the training set.

This is where the real value of the figure appears.

It shows that a chosen point does not just matter because of its own function value.
It matters because it changes the surrogate:

- the posterior mean is updated
- the uncertainty band changes
- and the set of plausible functions is altered

So the bottom row answers:

> **What kind of update does each acquisition rule induce in the surrogate?**

---

### Why this matters conceptually

This figure highlights a central idea in Bayesian Optimisation:

> acquisition functions are not only selecting points — they are selecting how the model will evolve.

That is why next-point selection matters so much.

Different acquisition rules may choose different locations, and those different choices can lead to different posterior updates.

So the distinction between PI and EI is not merely:
- one picks this point
- the other picks that point

It is also:
- one produces this kind of surrogate update
- the other produces that kind of surrogate update

That is a much deeper and more BO-relevant comparison.

---

### PI versus EI

This figure is especially useful because PI and EI are closely related but not identical.

Both are improvement-based rules, but they emphasise different notions of promise:

- **PI** focuses on the probability of beating the current best observed value
- **EI** focuses on the expected size of that improvement

So even when the selected points are close, the reasoning behind the choice is different.

This figure helps make that difference operational:
the two rules can lead to different new observations, and therefore to different posterior updates.

---

### Why the true objective is included

As in earlier tutorial figures, the true objective is plotted only for reference.

In a real optimisation problem, we would not know it everywhere.

It is included here so that we can see more clearly:

- how the selected point relates to the actual function
- and how the surrogate changes after new information is added

This makes it easier to interpret the consequences of PI and EI as decision rules.

---

### Why this is an important BO step

This cell moves the notebook from:

- **what acquisition functions look like**

to

- **what they actually do to the optimisation process**

That is a major conceptual step.

The decision rule is no longer just an abstract curve over the domain.
It has become an action that changes the dataset and therefore changes the surrogate model itself.

This is the essence of sequential Bayesian Optimisation.

---

### Key takeaway

> PI and EI do not merely score candidate points differently — they can also produce different updates to the Gaussian Process surrogate once their selected points are evaluated.

By comparing the GP before and after adding the PI-selected and EI-selected points, this figure shows that acquisition functions shape the optimisation process through the changes they induce in both:

- the posterior mean
- and the posterior uncertainty

In [ ]:
idx_pi = torch.argmax(pi_vals)
idx_ei = torch.argmax(ei_vals)

choices = {
    "PI": idx_pi,
    "EI": idx_ei,
}

results = {}
for name, idx in choices.items():
    x_new = x_dense[idx].unsqueeze(0)
    y_new = expensive_objective(x_new)
    x_train_new = torch.cat([x_train, x_new])
    y_train_new = torch.cat([y_train, y_new])

    mu_after, var_after, _ = gp_posterior(
        x_train_new, y_train_new, x_dense,
        lengthscale=0.8, variance=1.0, noise=1e-4
    )

    results[name] = {
        "idx": idx,
        "x_new": x_new,
        "y_new": y_new,
        "x_train_new": x_train_new,
        "y_train_new": y_train_new,
        "mu_after": mu_after,
        "sigma_after": torch.sqrt(var_after),
    }

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for col, name in enumerate(["PI", "EI"]):
    r = results[name]

    # Before adding selected point
    ax = axes[0, col]
    ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
    ax.plot(x_dense.numpy(), mu_gp.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
    ax.fill_between(
        x_dense.numpy(),
        (mu_gp - 2.0 * sigma_gp).numpy(),
        (mu_gp + 2.0 * sigma_gp).numpy(),
        alpha=0.2,
        label=r"$\mu(x)\pm 2\sigma(x)$"
    )
    ax.scatter(x_train.numpy(), y_train.numpy(), s=40, zorder=3, label="Observations")
    ax.scatter([float(r["x_new"])], [float(mu_gp[r["idx"]])], s=90, marker="*", zorder=4, label=f"{name}-selected point")
    ax.set_title(f"{name}: before adding selected point", fontsize=18, fontweight="bold")
    ax.set_xlabel("x", fontsize=22, fontweight="bold")
    ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
    style_ax(ax)

    # After adding selected point
    ax = axes[1, col]
    ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
    ax.plot(x_dense.numpy(), r["mu_after"].numpy(), linewidth=2.0, linestyle="--", label="GP mean")
    ax.fill_between(
        x_dense.numpy(),
        (r["mu_after"] - 2.0 * r["sigma_after"]).numpy(),
        (r["mu_after"] + 2.0 * r["sigma_after"]).numpy(),
        alpha=0.2,
        label=r"$\mu(x)\pm 2\sigma(x)$"
    )
    ax.scatter(r["x_train_new"].numpy(), r["y_train_new"].numpy(), s=40, zorder=3, label="Updated observations")
    ax.scatter([float(r["x_new"])], [float(r["y_new"])], s=90, marker="*", zorder=4, label=f"{name}-added point")
    ax.set_title(f"{name}: after adding selected point", fontsize=18, fontweight="bold")
    ax.set_xlabel("x", fontsize=22, fontweight="bold")
    ax.set_ylabel("f(x)", fontsize=22, fontweight="bold")
    style_ax(ax)

axes[0, 0].legend(prop={"size": 10, "weight": "bold"}, loc="lower center")
axes[0, 1].legend(prop={"size": 10, "weight": "bold"}, loc="lower center")
plt.tight_layout()
plt.show()

## 6. A simple Bayesian Optimisation loop with Expected Improvement

So far, the tutorial has shown the ingredients of Bayesian Optimisation one piece at a time:

- a Gaussian Process surrogate provides
  $\mu(x)$ and $\sigma(x)$,
- acquisition functions convert those quantities into a next-point decision,
- and different acquisition rules can lead to different posterior updates.

This cell now puts those ideas together into a simple **sequential Bayesian Optimisation loop**.

The aim is to show the basic logic of BO in executable form.

---

### What the code does

The function `run_bo_loop(...)` runs a short Bayesian Optimisation procedure starting from an initial dataset.

It takes as input:

- `x_init`, `y_init`: the initial observed points
- `x_grid`: the candidate search grid
- `n_steps`: the number of BO iterations
- the Gaussian Process hyperparameters

The loop then repeats the following steps:

1. fit a GP posterior to the current observations
2. compute the posterior standard deviation
3. find the current best observed value
4. compute the **Expected Improvement (EI)** acquisition across the candidate grid
5. exclude points that have already been observed
6. choose the next point by maximising EI
7. evaluate the true objective at that point
8. append the new observation to the dataset
9. store the updated dataset in the history

At the end, the function returns the full optimisation history.

So this is a compact implementation of the BO cycle:

> model → score candidates → choose next point → evaluate → update → repeat

---

### Why EI is used here

This loop uses **Expected Improvement** as the acquisition rule:

$$
x_{\text{next}} = \arg\max_x \text{EI}(x).
$$

That means the next point is chosen to maximise the expected amount by which it may improve on the best observed value so far.

So the decision rule is no longer based only on the GP mean.
It now uses both:

- the current prediction
- and the current uncertainty

This is what makes the loop Bayesian Optimisation rather than ordinary surrogate fitting.

---

### Why already observed points are masked out

The code computes the distance between each candidate grid point and the existing observations, then masks out points that have already been sampled.

This is done so that the acquisition function does not simply choose a point we already know.

Without this masking step, the optimisation loop could waste iterations by repeatedly selecting the same location.

So this is a practical implementation detail that ensures each BO step produces a genuinely new evaluation.

---

### What `history` stores

The variable `history` is a list of datasets collected over time.

Each element of `history` contains:

- the observed input locations at that stage
- the observed objective values at that stage

This allows us to inspect how the optimisation process evolves over successive iterations.

So `history` is what makes it possible to later plot:

- the GP posterior after several BO steps
- the growing observation set
- and the best observed value as optimisation progresses

---

### Why this function matters conceptually

This is the first point in the notebook where all the major ideas of Bayesian Optimisation are operating together inside a single loop.

It shows that BO is not just:
- a Gaussian Process,
- or an acquisition function,
- or a one-step decision.

It is a **repeated sequential process** in which the model and the dataset co-evolve.

Each new evaluation changes:

- the set of observations
- the GP posterior mean
- the GP uncertainty
- the acquisition landscape
- and therefore the next decision

That sequential dependence is the essence of Bayesian Optimisation.

---

### What this function is and is not

This is a deliberately simple BO loop.

It is:

- one-dimensional
- grid-based
- visual and interpretable
- intended for conceptual understanding

It is **not** yet:
- a high-dimensional optimisation routine
- a continuous acquisition optimiser
- or a production BO implementation

So the goal here is not computational sophistication.

The goal is clarity:

> to make the BO loop transparent enough that each step can be understood directly.

---

### Key takeaway

> This function implements the core Bayesian Optimisation loop: fit a GP, compute an acquisition function, choose the next point, evaluate the objective, update the data, and repeat.

That is the central procedural idea of BO.

Everything that follows in the notebook can now be interpreted as a visualisation of how this loop gradually improves the surrogate and the best value found so far.

In [ ]:
def run_bo_loop(x_init, y_init, x_grid, objective_fn=expensive_objective, n_steps=5, lengthscale=0.8, variance=1.0, noise=1e-4):
    x_obs = x_init.clone()
    y_obs = y_init.clone()

    history = [(x_obs.clone(), y_obs.clone())]

    for _ in range(n_steps):
        mu, var, _ = gp_posterior(x_obs, y_obs, x_grid, lengthscale=lengthscale, variance=variance, noise=noise)
        sigma = torch.sqrt(var)
        y_best = torch.min(y_obs)

        ei = ei_acquisition(mu, sigma, y_best, xi=0.0)

        distances = torch.min(torch.abs(x_grid[:, None] - x_obs[None, :]), dim=1).values
        mask = distances > 1e-6
        ei_masked = ei.clone()
        ei_masked[~mask] = -1.0

        idx = torch.argmax(ei_masked)
        x_new = x_grid[idx].unsqueeze(0)
        y_new = objective_fn(x_new)

        x_obs = torch.cat([x_obs, x_new])
        y_obs = torch.cat([y_obs, y_new])

        history.append((x_obs.clone(), y_obs.clone()))

    return history

## 7. Visualising the Bayesian Optimisation loop

The previous cell defined a simple Bayesian Optimisation loop based on **Expected Improvement (EI)**.

This cell now makes that loop visible.

Rather than looking only at the final result, we inspect the optimisation process at three different stages:

- the **initial state**
- **after 2 BO steps**
- **after 5 BO steps**

This allows us to see not only how the Gaussian Process surrogate evolves, but also how the **EI acquisition function** changes as new observations are added.

---

### What the code does

The code first runs the BO loop for 5 sequential iterations:

- starting from the initial observations
- fitting a GP at each step
- computing EI
- selecting the next point by maximising EI
- evaluating the true objective there
- and updating the dataset

The variable `history` stores the observation set after each stage of this process.

The figure then selects three snapshots from that history:

- step 0
- step 2
- step 5

and displays them in a $2\times 3$ layout.

---

### How the figure is organised

Each **column** corresponds to one stage of the BO process:

- left: initial state
- middle: after 2 BO steps
- right: after 5 BO steps

Each **row** shows a different part of the optimisation logic:

- **top row**: the GP surrogate
- **bottom row**: the EI acquisition function

So each column gives a complete “state of the optimiser” at that moment.

---

### Top row: how the surrogate evolves

In the top row, each panel shows:

- the true objective, for reference
- the GP posterior mean
- the uncertainty band
  $$
  \mu(x)\pm 2\sigma(x)
  $$
- the current observations
- and the **next EI-selected point** marked by a red star

This shows how the surrogate model changes as BO gathers more evaluations.

As more points are added:

- the posterior mean becomes more data-informed
- the uncertainty shrinks in more parts of the domain
- and the model’s estimate of promising regions becomes more refined

So the top row shows the learning side of Bayesian Optimisation.

---

### Bottom row: how the acquisition function evolves

In the bottom row, each panel shows the EI acquisition function at the same stage.

The red star marks the point that EI would choose next.

This is crucial, because it reveals the decision mechanism behind the loop.

At each stage, EI is computed from the current GP posterior, so its shape changes as the surrogate changes.

That means BO is not following a fixed policy.
Its next move depends on the current state of the model.

So the bottom row shows the decision side of Bayesian Optimisation.

---

### Why already observed points are hidden in the EI plot

The code masks previously sampled points in the acquisition plot by setting them to `NaN` for display and to a low value for selection.

This ensures that the optimiser does not waste evaluations by repeatedly selecting points that have already been observed.

So the EI curve is effectively showing only the useful candidate points that remain available for selection.

---

### How to interpret the full figure

Taken together, the six panels show the complete BO feedback loop:

1. the GP surrogate represents the current belief about the objective
2. EI uses that surrogate to score candidate points
3. the highest-EI point becomes the next evaluation
4. the new observation updates the surrogate
5. the acquisition landscape changes
6. the next BO decision is made from that updated state

This is the core logic of Bayesian Optimisation made explicit.

---

### Why this is an important figure

This is one of the most important visualisations in the tutorial.

It shows that Bayesian Optimisation is not just:

- a Gaussian Process
- plus an acquisition function
- plus some repeated evaluations

It is a **closed sequential loop** in which:

- the surrogate drives the acquisition
- the acquisition drives the next observation
- and the new observation reshapes the surrogate

That repeated coupling is what makes BO adaptive and data-efficient.

---

### Key takeaway

> Bayesian Optimisation is a sequential model-guided search process.

The top row shows how the GP surrogate improves as new observations are added.
The bottom row shows how the EI acquisition function changes in response and proposes the next evaluation.

Together, these panels make the BO loop visible:
the model and the decision rule evolve together over time.

In [ ]:
history = run_bo_loop(x_train, y_train, x_dense, n_steps=5)

panel_ids = [0, 2, 5]
panel_titles = ["Initial state", "After 2 BO steps", "After 5 BO steps"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True)

for col, (h_idx, title) in enumerate(zip(panel_ids, panel_titles)):
    x_obs, y_obs = history[h_idx]
    mu, var, _ = gp_posterior(x_obs, y_obs, x_dense, lengthscale=0.8, variance=1.0, noise=1e-4)
    sigma = torch.sqrt(var)
    y_best = torch.min(y_obs)

    ei = ei_acquisition(mu, sigma, y_best, xi=0.0)

    distances = torch.min(torch.abs(x_dense[:, None] - x_obs[None, :]), dim=1).values
    mask = distances > 1e-6

    ei_for_choice = ei.clone()
    ei_for_choice[~mask] = -1.0

    ei_for_plot = ei.clone()
    ei_for_plot[~mask] = float("nan")

    idx_next = torch.argmax(ei_for_choice)
    x_next = x_dense[idx_next]
    y_next_pred = mu[idx_next]

    # Top row: GP surrogate
    ax = axes[0, col]
    ax.plot(x_dense.numpy(), y_dense.numpy(), linewidth=2.0, label="True objective")
    ax.plot(x_dense.numpy(), mu.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
    ax.fill_between(
        x_dense.numpy(),
        (mu - 2.0 * sigma).numpy(),
        (mu + 2.0 * sigma).numpy(),
        alpha=0.2,
        label=r"$\mu(x)\pm 2\sigma(x)$"
    )
    ax.scatter(x_obs.numpy(), y_obs.numpy(), s=40, zorder=3, label="Observations")
    if h_idx < len(history) - 1:
        ax.scatter([float(x_next)], [float(y_next_pred)], s=90, marker="*", zorder=4, color="#D62728", label="Next EI point")
        ax.axvline(float(x_next), linewidth=1.6, linestyle="--", alpha=0.7)
    ax.set_title(title, fontsize=16, fontweight="bold")
    ax.set_xlabel("x", fontsize=18, fontweight="bold")
    ax.set_ylabel("f(x)", fontsize=18, fontweight="bold")
    style_ax(ax)

    # Bottom row: EI acquisition
    ax = axes[1, col]
    ax.plot(x_dense.numpy(), ei_for_plot.numpy(), linewidth=2.0, label="EI")
    if h_idx < len(history) - 1:
        ax.scatter([float(x_next)], [float(ei[idx_next])], s=90, marker="*", zorder=4, color="#D62728", label="Chosen next point")
        ax.axvline(float(x_next), linewidth=1.6, linestyle="--", alpha=0.7)
    ax.set_title(f"EI acquisition: {title}", fontsize=16, fontweight="bold")
    ax.set_xlabel("x", fontsize=18, fontweight="bold")
    ax.set_ylabel("EI", fontsize=18, fontweight="bold")
    style_ax(ax)

axes[0, 0].legend(prop={"size": 10, "weight": "bold"})
axes[1, 0].legend(prop={"size": 10, "weight": "bold"})

plt.tight_layout()
plt.show()

## 8. The best value found so far does not improve at every BO step

The previous figure showed how the Gaussian Process surrogate and the EI acquisition function evolve over the optimisation process.

This final cell looks at BO from a simpler performance perspective:

> **How does the best objective value found so far change over successive BO iterations?**

This is one of the most natural ways to summarise whether Bayesian Optimisation is making progress.

---

### What the code does

The variable `history` stores the observed dataset after each stage of the BO loop.

For each stage, the code computes

$$
\min(y_{\text{obs}}),
$$

that is, the **best observed objective value so far**.

These values are collected into the list `best_vals` and plotted against the BO iteration number.

So the curve is not showing the value of the *newest* point at each iteration.
Instead, it is showing the **running best-so-far value**.

That means the curve can only:
- stay the same
- or improve downward

since we are solving a minimisation problem.

---

### How to interpret the figure

Each point on the curve answers the question:

> **After this many BO steps, what is the lowest objective value that has been observed up to now?**

If the curve drops, that means a new best point has been discovered.

If the curve stays flat, that means the most recent evaluation did **not** beat the best value already found, even though it may still have provided useful information.

So the plot is a compact summary of optimisation progress over time.

---

### Why BO iteration 1 and 2 may not improve the best value

A very important point is that Bayesian Optimisation is **not guaranteed to improve the best observed value at every single iteration**.

This is especially true in early steps.

If BO iteration 1 and 2 do not produce a lower best value, that is completely normal.

Why?

Because BO is not only trying to exploit what already looks best.
It is also trying to **learn about uncertain regions**.

So an early evaluation may be selected because:

- the uncertainty is large there,
- the point has high expected improvement,
- or the surrogate needs more information in that part of the domain,

even if the actual observed value there does not beat the current best.

In other words:

> a BO step can be valuable even when it does not immediately improve the best observed value.

It may still reduce uncertainty, improve the surrogate, and lead to better choices later.

---

### Why this makes sense in terms of EI

This is particularly important for **Expected Improvement**.

EI does not choose points that are guaranteed to improve the objective.
It chooses points with high **expected** improvement under the current surrogate.

That expectation includes uncertainty.

So a point can have high EI because:

- it looks promising under the mean,
- or it is uncertain enough that a meaningful improvement is still plausible.

If the actual evaluation then turns out not to beat the current best, that does not mean EI failed.
It means the model explored a point that was worth checking under the information available at the time.

That is the nature of sequential decision-making under uncertainty.

---

### Why flat steps are not wasted steps

A flat segment in the best-so-far curve does **not** mean the BO iteration was useless.

Even if the newest point does not improve the running best value, it can still:

- refine the GP posterior mean
- reduce uncertainty in a strategically important region
- reshape the acquisition landscape
- and improve future next-point choices

So the BO process should not be judged only by whether every step immediately lowers the best value.

A more accurate interpretation is:

> some steps directly improve the best value, while others improve the model so that later steps can do so.

---

### What this figure tells us overall

This plot captures one of the key features of Bayesian Optimisation:

- progress is often **non-uniform**
- improvements may happen in jumps rather than smoothly at every step
- and information-gathering steps may precede value-improving steps

That is exactly what we should expect from an optimisation method that balances:

- **exploitation** of currently good-looking regions
- with **exploration** of uncertain regions

---

### Key takeaway

> The best observed value in Bayesian Optimisation does not need to improve at every iteration.

If BO iteration 1 and 2 do not yield better values, that is normal:
those evaluations may still have been useful because they improved the surrogate model and reduced uncertainty, which can enable better optimisation steps later.

So this curve should be read as a record of **best-so-far progress**, not as a requirement that every BO step must immediately beat the current optimum.

In [ ]:
best_vals = [float(torch.min(y_obs)) for (_, y_obs) in history]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(np.arange(len(best_vals)), best_vals, "-o", linewidth=2.5)
ax.set_title("Best observed value improves over BO iterations", fontsize=18, fontweight="bold")
ax.set_xlabel("BO iteration", fontsize=22, fontweight="bold")
ax.set_ylabel("Best observed value", fontsize=22, fontweight="bold")
style_ax(ax)
plt.show()

## 9. A more challenging closing example

To close Part 3, we now apply the same Bayesian Optimisation loop to a more challenging one-dimensional objective.

This function is designed to be more interesting than the earlier running example:

- it contains **multiple scales**,
- it has a **broad basin** that can look attractive early on,
- and it also contains sharper structure that is harder to infer from only a few initial evaluations.

This makes it a useful final example, because it requires the full Part 3 pipeline to work together:

- modelling an unknown objective,
- representing uncertainty,
- building a Gaussian Process surrogate,
- and using an acquisition function to guide sequential evaluation.

---

### What the code does

The code first defines a new objective, `history_shaped_objective(x)`, which is deliberately less benign than the earlier toy function.

It then:

- creates a dense evaluation grid over
  $$
  [-3.5,\;3.5],
  $$
- samples a small initial design of 6 points across that interval,
- and runs the same Bayesian Optimisation loop for 8 EI-guided iterations.

The resulting optimisation history is then visualised at five stages:

- initial state
- after 2 BO steps
- after 4 BO steps
- after 6 BO steps
- after 8 BO steps

The figure is arranged as a $2\times5$ grid.

As before:

- the **top row** shows the GP surrogate:
  - true objective
  - GP mean
  - uncertainty band
  - observations
  - next EI-selected point
- the **bottom row** shows the corresponding **EI acquisition function**

So this is the same BO machinery as before, now tested on a more demanding objective.

---

### Why this is a good closing example

This example is useful because it makes Bayesian Optimisation feel more realistic.

The function is no longer something that can be understood almost immediately from a few evenly spaced observations.

Instead:

- some regions look promising early on,
- others remain uncertain,
- and the optimiser must use the surrogate and acquisition function together to decide where more information is worth gathering.

That is exactly the setting where Bayesian Optimisation becomes meaningful.

---

### How to interpret the figure

The most important thing to watch is not whether the GP becomes globally perfect everywhere.

Instead, the important questions are:

- where does EI direct new evaluations?
- how does the observation set expand over time?
- and does the optimiser move toward the genuinely promising region of the objective?

As the BO steps accumulate, the GP surrogate changes, the uncertainty contracts in sampled regions, and the EI acquisition function becomes increasingly concentrated around parts of the domain that still look useful for improvement.

So the figure is showing the BO loop behaving as it should:
the surrogate and the decision rule are co-evolving as information is gathered.

---

### Why it is okay if the GP is still inaccurate in some regions

An important point to state explicitly is this:

> even if the GP mean is still quite different from the true function in some parts of the domain after 8 BO steps, that is not a failure of Bayesian Optimisation.

Why not?

Because BO is **not trying to reconstruct the whole function perfectly**.

Its goal is to use a limited evaluation budget to find good minima efficiently.

So if the GP remains inaccurate in regions that do **not** contain the real minimum, that is usually not a problem.

In other words:

- we do **not** require the surrogate to be globally accurate everywhere,
- we mainly care that it becomes informative enough in the regions relevant to optimisation.

So for this closing example, the correct question is not:

> “Does the GP match the true objective perfectly after 8 steps?”

The correct question is:

> “Has the optimisation process been guided toward the truly important low-value region?”

That is the more relevant standard for BO.

---

### Why this matters conceptually

This is a very important distinction between:

- **surrogate modelling for global function reconstruction**
- and **surrogate modelling for optimisation**

If the objective were full-function recovery, then a poor GP fit in any region would be a serious issue.

But in Bayesian Optimisation, what matters most is whether the surrogate is good enough to support strong decisions about where to sample next.

So this closing figure makes a mature Part 3 point:

> a surrogate can be imperfect globally and still be useful for optimisation.

That is one of the reasons uncertainty-aware model-guided search is so powerful.

---

### What this example shows about Part 3 as a whole

This final example brings together the full progression of the part:

- from modelling an unknown function,
- to recognising that uncertainty matters,
- to building a principled probabilistic surrogate with Gaussian Processes,
- to using that surrogate in a sequential Bayesian Optimisation loop.

So this is not just another GP figure.

It is the point where the individual concepts of Part 3 become a working optimisation system.

---

### Key takeaway

> The goal of Bayesian Optimisation is not to make the Gaussian Process perfectly match the true objective everywhere. The goal is to use the surrogate and its uncertainty to guide evaluations toward the genuinely important low-value region.

So even if the GP remains quite different from the true function in some regions after 8 BO steps, that is acceptable — and even expected — if those regions do not contain the real minimum.

That is what makes this a strong closing example for Part 3:
it shows the full uncertainty-aware optimisation pipeline working on a function where global perfection is unnecessary, but good sequential decisions really matter.

In [ ]:
def history_shaped_objective(x):
    return (
        0.035 * (x + 2.6) ** 2
        - 0.60 * torch.exp(-0.5 * ((x + 1.15) / 0.70) ** 2)
        - 1.10 * torch.exp(-0.5 * ((x - 2.15) / 0.22) ** 2)
        + 0.11 * torch.sin(3.4 * x)
        + 0.07 * torch.cos(7.6 * x)
        + 0.12
    )

x_dense_hard = torch.linspace(-3.5, 3.5, 500)
y_dense_hard = history_shaped_objective(x_dense_hard)

x_init_hard = torch.linspace(-3.5, 3.5, 6)
y_init_hard = history_shaped_objective(x_init_hard)

history_hard = run_bo_loop(
    x_init_hard,
    y_init_hard,
    x_dense_hard,
    objective_fn=history_shaped_objective,
    n_steps=8,
    lengthscale=0.8,
    variance=1.0,
    noise=1e-4,
)

panel_ids_hard = [0, 2, 4, 6, 8]
panel_titles_hard = [
    "Initial state",
    "After 2 BO steps",
    "After 4 BO steps",
    "After 6 BO steps",
    "After 8 BO steps",
]

fig, axes = plt.subplots(2, 5, figsize=(26,9), sharex=True)

for col, (h_idx, title) in enumerate(zip(panel_ids_hard, panel_titles_hard)):
    x_obs, y_obs = history_hard[h_idx]
    mu, var, _ = gp_posterior(x_obs, y_obs, x_dense_hard, lengthscale=0.8, variance=1.0, noise=1e-4)
    sigma = torch.sqrt(var)
    y_best = torch.min(y_obs)

    ei = ei_acquisition(mu, sigma, y_best, xi=0.0)

    distances = torch.min(torch.abs(x_dense_hard[:, None] - x_obs[None, :]), dim=1).values
    mask = distances > 1e-6

    ei_for_choice = ei.clone()
    ei_for_choice[~mask] = -1.0

    ei_for_plot = ei.clone()
    ei_for_plot[~mask] = float("nan")

    idx_next = torch.argmax(ei_for_choice)
    x_next = x_dense_hard[idx_next]
    y_next_pred = mu[idx_next]

    # Top row: GP surrogate
    ax = axes[0, col]
    ax.plot(x_dense_hard.numpy(), y_dense_hard.numpy(), linewidth=2.0, label="True objective")
    ax.plot(x_dense_hard.numpy(), mu.numpy(), linewidth=2.0, linestyle="--", label="GP mean")
    ax.fill_between(
        x_dense_hard.numpy(),
        (mu - 2.0 * sigma).numpy(),
        (mu + 2.0 * sigma).numpy(),
        alpha=0.2,
        label=r"$\mu(x)\pm 2\sigma(x)$"
    )
    ax.scatter(x_obs.numpy(), y_obs.numpy(), s=40, zorder=3, label="Observations")
    if h_idx < len(history_hard) - 1:
        ax.scatter([float(x_next)], [float(y_next_pred)], s=90, marker="*", zorder=4, color="#D62728", label="Next EI point")
        ax.axvline(float(x_next), linewidth=1.6, linestyle="--", alpha=0.7)
    ax.set_title(title, fontsize=16, fontweight="bold")
    ax.set_xlabel("x", fontsize=18, fontweight="bold")
    ax.set_ylabel("f(x)", fontsize=18, fontweight="bold")
    style_ax(ax)

    # Bottom row: EI acquisition
    ax = axes[1, col]
    ax.plot(x_dense_hard.numpy(), ei_for_plot.numpy(), linewidth=2.0, label="EI")
    if h_idx < len(history_hard) - 1:
        ax.scatter([float(x_next)], [float(ei[idx_next])], s=90, marker="*", zorder=4, color="#D62728", label="Chosen next point")
        ax.axvline(float(x_next), linewidth=1.6, linestyle="--", alpha=0.7)
    ax.set_title(f"EI acquisition: {title}", fontsize=16, fontweight="bold")
    ax.set_xlabel("x", fontsize=18, fontweight="bold")
    ax.set_ylabel("EI", fontsize=18, fontweight="bold")
    style_ax(ax)

axes[0, 0].legend(prop={"size": 9, "weight": "bold"})
axes[1, 0].legend(prop={"size": 9, "weight": "bold"})
plt.tight_layout()
plt.show()

## 🧭 Closing Remarks

In this tutorial, we moved from understanding Gaussian Process surrogates to using them for **sequential decision-making**.

That is the central transition from surrogate modelling to **Bayesian Optimisation**.

In the earlier tutorials of Part 3, the main questions were:

- why an unknown objective should be modelled,
- why uncertainty matters,
- and how a Gaussian Process provides a principled predictive mean and uncertainty band.

Here, the question changed.

Instead of asking only

> *What does the surrogate believe?*

we asked

> *Given the surrogate, where should we evaluate next?*

That is the defining question of Bayesian Optimisation.

---

A clear structure emerged throughout the notebook.

First, we saw that choosing the next point by minimising the GP mean alone is too greedy.
That rule uses only current prediction and ignores uncertainty, so it corresponds to pure exploitation.

We then introduced **acquisition functions** as the missing decision layer.

These functions do not model the objective themselves.
Instead, they use the GP posterior to decide which candidate point is most worth evaluating next.

In particular, we studied:

- **LCB**, which balances low mean and high uncertainty,
- **PI**, which asks how likely a point is to improve on the best value already found,
- and **EI**, which asks how much improvement is expected on average.

That shift was important, because it turned the GP from a passive predictive model into an active guide for data collection.

---

The later figures showed the real significance of this idea.

Acquisition functions are not just abstract scoring curves.
They determine which point is sampled next, and that choice changes:

- the observation set,
- the GP posterior mean,
- the GP uncertainty,
- and the next acquisition landscape.

So Bayesian Optimisation is not a one-shot procedure.
It is a **closed sequential loop**:

1. fit the surrogate
2. compute the acquisition
3. choose the next point
4. evaluate the objective
5. update the surrogate
6. repeat

This loop is the operational heart of BO.

---

The notebook also highlighted an important subtlety:

> a BO iteration does not need to improve the best observed value immediately in order to be useful.

Some evaluations improve the best value directly.
Others improve the model first by reducing uncertainty or clarifying promising regions.
Those information-gathering steps can be what enable later improvement.

That is why Bayesian Optimisation often progresses in uneven jumps rather than in a smooth, monotonic way at every single iteration.

---

The final closing example brought all of Part 3 together.

On a more challenging objective, we saw that the GP surrogate may still remain imperfect in some regions even after several BO steps.
But that is not a failure.

The goal of Bayesian Optimisation is **not** to reconstruct the entire function perfectly.
The goal is to use limited evaluations efficiently enough to identify genuinely promising low-value regions.

So the mature lesson of Part 3 is this:

> a surrogate can be globally imperfect and still be highly useful for optimisation.

What matters is not perfect reconstruction everywhere, but whether the surrogate and acquisition function together are good enough to guide strong sequential decisions.

---

This completes the conceptual arc of Part 3:

- **Tutorial 1**: why model an unknown function
- **Tutorial 2**: why uncertainty must be represented explicitly
- **Tutorial 3**: how Gaussian Processes provide a principled probabilistic surrogate
- **Tutorial 4**: how acquisition functions turn that surrogate into a sequential optimisation strategy

So by the end of this part, the full Bayesian Optimisation pipeline is now in place:

- model the unknown objective,
- quantify uncertainty,
- use a GP surrogate,
- and choose new evaluations through acquisition-driven search.

---

This also sets up the natural transition into the next part of the repository.

In Part 4, we will no longer build everything from first principles in the same way.
Instead, we will begin using **BoTorch** to implement Bayesian Optimisation workflows more practically and at a higher level.

That means the focus will shift from:

- *what Bayesian Optimisation is*
to
- *how to use modern BO tools effectively in practice*.

So if Part 3 built the conceptual foundation, Part 4 will build the implementation layer on top of it.

---

**Final takeaway**

> Bayesian Optimisation works because a probabilistic surrogate model does more than predict the objective. It also quantifies uncertainty, and that uncertainty can be used to decide where information is most valuable next.

That is the core idea that makes model-guided optimisation possible.